### Import Libraries

In [2]:
import pandas as pd
import numpy as np
import fastf1
import requests
from pathlib import Path

### Create Project Directories

In [3]:
# Create folders if they don't exist
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../data/raw").mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


### Load Historical Kaggle Data

In [4]:
DATA_PATH = "../data/raw/"

races = pd.read_csv(DATA_PATH + "races.csv")
results = pd.read_csv(DATA_PATH + "results.csv") 
drivers = pd.read_csv(DATA_PATH + "drivers.csv") 
constructors = pd.read_csv(DATA_PATH + "constructors.csv") 
qualifying = pd.read_csv(DATA_PATH + "qualifying.csv") 
circuits = pd.read_csv(DATA_PATH + "circuits.csv")

### Quick Inspection

In [5]:
print("Races:", races.shape)
print("Results:", results.shape)
print("Drivers:", drivers.shape)
print("Constructors:", constructors.shape)
print("Qualifying:", qualifying.shape)

Races: (1125, 18)
Results: (26759, 18)
Drivers: (861, 9)
Constructors: (212, 5)
Qualifying: (10494, 9)


### Keep Modern Era Only (2018+)

In [6]:
modern_races = races[races['year'] >= 2018]
modern_race_ids = modern_races["raceId"].unique()

results = results[results["raceId"].isin(modern_race_ids)]
qualifying = qualifying[qualifying["raceId"].isin(modern_race_ids)]

print("Modern Races:", modern_races.shape)
print("Modern Results:", results.shape)
print("Modern Qualifying:", qualifying.shape)

Modern Races: (149, 18)
Modern Results: (2979, 18)
Modern Qualifying: (2976, 9)


### Merge Core Tables

In [7]:
df = results.merge( modern_races, on="raceId", how="left" )
print("Merged DataFrame:", df.shape)
df = df.merge( drivers[["driverId", "driverRef", "surname"]], on="driverId", how="left" )
print("Merged with Drivers:", df.shape)
df = df.merge(constructors[["constructorId", "name"]], on="constructorId", how="left" )
print("Merged with Constructors:", df.shape)
df = df.merge( circuits[["circuitId", "circuitRef", "name"]], on="circuitId", how="left" )
print("Merged with Circuits:", df.shape)
df.columns

Merged DataFrame: (2979, 35)
Merged with Drivers: (2979, 37)
Merged with Constructors: (2979, 38)
Merged with Circuits: (2979, 40)


Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time_x',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name_x',
       'date', 'time_y', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time',
       'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date',
       'sprint_time', 'driverRef', 'surname', 'name_y', 'circuitRef', 'name'],
      dtype='object')

### Merge Qualifying Data

In [8]:
quali_cols = ["raceId", "driverId", "position"]
qualifying = qualifying[quali_cols].copy()

qualifying.rename(columns = {"position": "quali_position"}, inplace=True)
df = df.merge(qualifying, on=["raceId", "driverId"], how="left")

print("Final DataFrame:", df.shape)
df.columns

Final DataFrame: (2979, 41)


Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time_x',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name_x',
       'date', 'time_y', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time',
       'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date',
       'sprint_time', 'driverRef', 'surname', 'name_y', 'circuitRef', 'name',
       'quali_position'],
      dtype='object')

### Select Important Columns

In [9]:
df = df[[
    "raceId", "year", "round", "date", "name_x", "circuitRef", 
    "driverId", "driverRef", "surname", 
    "constructorId", "name_y", "grid", 
    "positionOrder", "points", "statusId", 
    "quali_position" 
    ]
]
df.head()

,raceId,year,round,date,name_x,circuitRef,driverId,driverRef,surname,constructorId,name_y,grid,positionOrder,points,statusId,quali_position
0,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,20,vettel,Vettel,6,Ferrari,3,1,25.0,1,3.0
1,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,1,hamilton,Hamilton,131,Mercedes,1,2,18.0,1,1.0
2,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,8,raikkonen,Räikkönen,6,Ferrari,2,3,15.0,1,2.0
3,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,817,ricciardo,Ricciardo,9,Red Bull,8,4,12.0,1,5.0
4,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,4,alonso,Alonso,1,McLaren,10,5,10.0,1,11.0


In [10]:
# rename columns
df.columns =[
              [ "race_id", "year", "round", 
              "race_date", "race_name", "circuit_id", 
              "driver_id", "driver_code", "driver_name", 
              "constructor_id", "constructor_name", "grid_position", 
              "finish_position", "points", "status_id", "quali_position" ]
]
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position
0,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,20,vettel,Vettel,6,Ferrari,3,1,25.0,1,3.0
1,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,1,hamilton,Hamilton,131,Mercedes,1,2,18.0,1,1.0
2,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,8,raikkonen,Räikkönen,6,Ferrari,2,3,15.0,1,2.0
3,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,817,ricciardo,Ricciardo,9,Red Bull,8,4,12.0,1,5.0
4,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,4,alonso,Alonso,1,McLaren,10,5,10.0,1,11.0


In [11]:
df['winner'] = np.where(df['finish_position'] == 1,1,0)
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner
0,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,20,vettel,Vettel,6,Ferrari,3,1,25.0,1,3.0,1
1,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,1,hamilton,Hamilton,131,Mercedes,1,2,18.0,1,1.0,0
2,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,8,raikkonen,Räikkönen,6,Ferrari,2,3,15.0,1,2.0,0
3,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,817,ricciardo,Ricciardo,9,Red Bull,8,4,12.0,1,5.0,0
4,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,4,alonso,Alonso,1,McLaren,10,5,10.0,1,11.0,0


In [12]:
df.isnull().sum()

race_id             0
year                0
round               0
race_date           0
race_name           0
circuit_id          0
driver_id           0
driver_code         0
driver_name         0
constructor_id      0
constructor_name    0
grid_position       0
finish_position     0
points              0
status_id           0
quali_position      3
winner              0
dtype: int64

In [13]:
df["quali_position"] = df["quali_position"].fillna(df["grid_position"])
df.isnull().sum()

race_id             0
year                0
round               0
race_date           0
race_name           0
circuit_id          0
driver_id           0
driver_code         0
driver_name         0
constructor_id      0
constructor_name    0
grid_position       0
finish_position     0
points              0
status_id           0
quali_position      3
winner              0
dtype: int64

In [14]:
df.dropna(inplace=True)
df.isnull().sum()

race_id             0
year                0
round               0
race_date           0
race_name           0
circuit_id          0
driver_id           0
driver_code         0
driver_name         0
constructor_id      0
constructor_name    0
grid_position       0
finish_position     0
points              0
status_id           0
quali_position      0
winner              0
dtype: int64

In [16]:
fastf1.Cache.enable_cache("../data/cache")
completed_races_2026 = [ "Australia", "China", "Japan", "Bahrain", "Miami", "Monaco" ]


### Download 2026 Race Results

In [17]:
all_2026_results = []

for race in completed_races_2026:
    try:
        print(f"Loading {race} 2026...")
        session = fastf1.get_session(2026, race, "R")
        session.load()

        results_2026 = session.results.copy()

        results_2026["race_name"] = race
        results_2026["year"] = 2026
        
        all_2026_results.append(results_2026)
    
    except Exception as e:
        print(f"Failed for {race} : {e}")

Loading Australia 2026...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

Loading China 2026...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

Loading Japan 2026...


...

events      WARNING 	Correcting user input 'Bahrain' to 'São Paulo Grand Prix'
core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching l

Loading Miami 2026...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

Loading Monaco 2026...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

### Combine FastF1 Results

In [20]:
f1_2026 = pd.concat(all_2026_results)
print(f1_2026.columns)
f1_2026.head()

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps',
       'race_name', 'year'],
      dtype='object')


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,race_name,year
63,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,...,1.0,NaT,NaT,NaT,0 days 01:23:06.801000,Finished,25.0,58.0,Australia,2026
12,12,K ANTONELLI,ANT,antonelli,Mercedes,00D7B6,mercedes,Kimi,Antonelli,Kimi Antonelli,...,2.0,NaT,NaT,NaT,0 days 00:00:02.974000,Finished,18.0,58.0,Australia,2026
16,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,...,4.0,NaT,NaT,NaT,0 days 00:00:15.519000,Finished,15.0,58.0,Australia,2026
44,44,L HAMILTON,HAM,hamilton,Ferrari,ED1131,ferrari,Lewis,Hamilton,Lewis Hamilton,...,7.0,NaT,NaT,NaT,0 days 00:00:16.144000,Finished,12.0,58.0,Australia,2026
1,1,L NORRIS,NOR,norris,McLaren,F47600,mclaren,Lando,Norris,Lando Norris,...,6.0,NaT,NaT,NaT,0 days 00:00:51.741000,Finished,10.0,58.0,Australia,2026


In [27]:
f1_2026 = f1_2026.reset_index(drop=True)
f1_2026 = f1_2026[ [ "Abbreviation", "FullName", "TeamName", "GridPosition", "Position", "Points", "race_name", "year" ] ]
# Rename 2026 Columns
f1_2026.rename(columns={
    "Abbreviation": "driver_code",
    "FullName": "driver_name",
    "TeamName": "constructor_name",
    "GridPosition": "grid_position",
    "Position": "finish_position",
    "Points": "points"
}, inplace=True)
print(f1_2026.columns)
f1_2026.head()

Index(['driver_code', 'driver_name', 'constructor_name', 'grid_position',
       'finish_position', 'points', 'race_name', 'year'],
      dtype='object')


,driver_code,driver_name,constructor_name,grid_position,finish_position,points,race_name,year
0,RUS,George Russell,Mercedes,1.0,1.0,25.0,Australia,2026
1,ANT,Kimi Antonelli,Mercedes,2.0,2.0,18.0,Australia,2026
2,LEC,Charles Leclerc,Ferrari,4.0,3.0,15.0,Australia,2026
3,HAM,Lewis Hamilton,Ferrari,7.0,4.0,12.0,Australia,2026
4,NOR,Lando Norris,McLaren,6.0,5.0,10.0,Australia,2026


In [28]:
f1_2026["finish_position"] = pd.to_numeric(f1_2026["finish_position"], errors="coerce")
f1_2026["winner"] = np.where( f1_2026["finish_position"] == 1, 1, 0 )
f1_2026.head()

,driver_code,driver_name,constructor_name,grid_position,finish_position,points,race_name,year,winner
0,RUS,George Russell,Mercedes,1.0,1.0,25.0,Australia,2026,1
1,ANT,Kimi Antonelli,Mercedes,2.0,2.0,18.0,Australia,2026,0
2,LEC,Charles Leclerc,Ferrari,4.0,3.0,15.0,Australia,2026,0
3,HAM,Lewis Hamilton,Ferrari,7.0,4.0,12.0,Australia,2026,0
4,NOR,Lando Norris,McLaren,6.0,5.0,10.0,Australia,2026,0


### Save Raw Combined Data

In [29]:
df.to_csv("../data/processed/historical_f1_data.csv", index=False)

f1_2026.to_csv("../data/processed/f1_2026_results.csv", index=False)
print("Files saved successfully.")

Files saved successfully.


### Final Dataset Check

In [30]:
print(df.head())

print("\nHistorical shape:", df.shape)
print("2026 shape:", f1_2026.shape)

  race_id  year round   race_date              race_name   circuit_id  \
0     989  2018     1  2018-03-25  Australian Grand Prix  albert_park   
1     989  2018     1  2018-03-25  Australian Grand Prix  albert_park   
2     989  2018     1  2018-03-25  Australian Grand Prix  albert_park   
3     989  2018     1  2018-03-25  Australian Grand Prix  albert_park   
4     989  2018     1  2018-03-25  Australian Grand Prix  albert_park   

  driver_id driver_code driver_name constructor_id constructor_name  \
0        20      vettel      Vettel              6          Ferrari   
1         1    hamilton    Hamilton            131         Mercedes   
2         8   raikkonen   Räikkönen              6          Ferrari   
3       817   ricciardo   Ricciardo              9         Red Bull   
4         4      alonso      Alonso              1          McLaren   

  grid_position finish_position points status_id quali_position winner  
0             3               1   25.0         1            3

### Export Combined Dataset

- for Feature Engineering

In [31]:
combined = pd.concat( [ df[[ "year", "race_name", "driver_name",
                             "constructor_name", "grid_position", 
                             "finish_position", "points", "winner" ]], 
                        f1_2026[[ "year", "race_name", "driver_name", 
                                 "constructor_name", "grid_position", "finish_position", 
                                 "points", "winner" ]] ], ignore_index=True ) 

combined.to_csv( "../data/processed/f1_combined_dataset.csv", index=False )